# Naive RAG Base

## 一、RAG-检索增强生成

### 1.1 LLM的局限性

1. LLM的知识不是实时的，不具备知识更新.
2. LLM可能不知道你私有的领域/业务知识.
3. LLM有时会在回答中生成看似合理但实际上是错误的信息.

### 1.2 为什么会用到RAG

1、提高准确性: 通过检索相关的信息，RAG可以提高生成文本的准确性。

2、减少训练成本：与需要大量数据来训练的大型生成模型相比，RAG可以通过检索机制来减少所需的训练数据量，从而降低训练成本。

3、适应性强：RAG模型可以适应新的或不断变化的数据。由于它们能够检索最新的信息，因此在新数据和事件出现时，它们能够快速适应并生成相关的文本。

### 1.3 RAG概念

RAG（Retrieval Augmented Generation）顾名思义，通过**检索**的方法来增强**生成模型**的能力。

<video src="./img/RAG.mp4" controls="controls" width=800px style="margin-left: 0px"></video>

### 1.4 RAG类比

类比：你可以把这个过程想象成开卷考试。让LLM先翻书，再回答问题。

## 二、Naive RAG Pipeline

<img src="./img/RAG工作流程图解.jpg" width="100%">

<img src="./img/RAG系统搭建流程.png" width="100%">

## 三、向量检索

### 3.1 检索的方式有那些

列举两种: 

1、关键字搜索：通过用户输入的关键字来查找文本数据。

2、语义搜索：不仅考虑关键词的匹配，还考虑词汇之间的语义关系，以提供更准确的搜索结果。


### 3.2 向量与Embeddings的定义

在数学中，向量（也称为欧几里得向量、几何向量），指具有大小（magnitude）和方向的量。它可以形象化地表示为带箭头的线段。箭头所指：代表向量的方向；线段长度：代表向量的大小。

1. 将文本转成一组浮点数：每个下标 $i$，对应一个维度
2. 整个数组对应一个 $n$ 维空间的一个点，即**文本向量**又叫 Embeddings
3. 向量之间可以计算距离，距离远近对应**语义相似度**大小



<img src="./img/embeddings.png" width ="80%">

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
client = OpenAI()

def get_embeddings(texts, model="text-embedding-3-large"):
    #  texts 是一个包含要获取嵌入表示的文本的列表，
    #  model 则是用来指定要使用的模型的名称
    #  生成文本的嵌入表示。结果存储在data中。
    data = client.embeddings.create(input=texts, model=model).data
    # print(data)
    # 返回了一个包含所有嵌入表示的列表
    return [x.embedding for x in data]


test_query = ["我爱你"]

vec = get_embeddings(test_query)
#  "我爱你" 文本嵌入表示的列表。
print(vec)
#  "我" 文本的嵌入表示。
print(vec[0])
#  "我" 文本的嵌入表示的维度。3072
print(len(vec[0]))

### 3.3 向量间的相似度计算 

<img src="./img/sim.png" width ="60%">

In [7]:
from openai import OpenAI
from dotenv import load_dotenv
import numpy as np
from numpy import dot
from numpy.linalg import norm
load_dotenv()
client = OpenAI()

def cos_sim(a, b):
    '''余弦距离 -- 越大越相似'''
    return dot(a, b)/(norm(a)*norm(b))

def l2(a, b):
    '''欧式距离 -- 越小越相似'''
    x = np.asarray(a)-np.asarray(b)
    return norm(x)

def get_embeddings(texts, model="text-embedding-3-large"):
    #  texts 是一个包含要获取嵌入表示的文本的列表，
    #  model 则是用来指定要使用的模型的名称
    #  生成文本的嵌入表示。结果存储在data中。
    data = client.embeddings.create(input=texts, model=model).data
    # print(data)
    # 返回了一个包含所有嵌入表示的列表
    return [x.embedding for x in data]


# 且能支持跨语言
# query = "global conflicts"
query = "国际争端"
documents = [
    "联合国就苏丹达尔富尔地区大规模暴力事件发出警告",
    "土耳其、芬兰、瑞典与北约代表将继续就瑞典“入约”问题进行谈判",
    "日本岐阜市陆上自卫队射击场内发生枪击事件 3人受伤",
    "国家游泳中心（水立方）：恢复游泳、嬉水乐园等水上项目运营",
    "我国首次在空间站开展舱外辐射生物学暴露实验",
]

query_vec = get_embeddings([query])[0]

doc_vecs = get_embeddings(documents)


print("Cosine distance:")
print(cos_sim(query_vec, query_vec))
for vec in doc_vecs:
    print(cos_sim(query_vec, vec))


print("\nEuclidean distance:")
print(l2(query_vec, query_vec))
for vec in doc_vecs:
    print(l2(query_vec, vec))

Cosine distance:
0.9999999999999999
0.2370090408983918
0.28231843637704235
0.1152975379025242
0.14936589374726902
0.13168852079837604

Euclidean distance:
0.0
1.2353063919692502
1.1980664037889215
1.330189849783533
1.3043267095981308
1.3178098771928697


### 3.4 向量数据库

In [ ]:
from pdfminer.high_level import extract_pages
from pdfminer.layout import LTTextContainer
from openai import OpenAI
import chromadb
from chromadb.config import Settings
from dotenv import load_dotenv
load_dotenv()
client = OpenAI()

def get_embeddings(texts, model="text-embedding-ada-002"):
    print('11111222233333')
    '''封装 OpenAI 的 Embedding 模型接口'''
    data = client.embeddings.create(input=texts, model=model).data
    return [x.embedding for x in data]

def extract_text_from_pdf(filename, page_numbers=None, min_line_length=1):
    '''从 PDF 文件中（按指定页码）提取文字'''
    paragraphs = []
    buffer = ''
    full_text = ''
    # 提取全部文本
    for i, page_layout in enumerate(extract_pages(filename)):
        # 如果指定了页码范围，跳过范围外的页
        if page_numbers is not None and i not in page_numbers:
            continue
        for element in page_layout:
            if isinstance(element, LTTextContainer):
                full_text += element.get_text() + '\n'
    # 按空行分隔，将文本重新组织成段落
    lines = full_text.split('\n')
    for text in lines:
        if len(text) >= min_line_length:
            buffer += (' '+text) if not text.endswith('-') else text.strip('-')
        elif buffer:
            paragraphs.append(buffer)
            buffer = ''
    if buffer:
        paragraphs.append(buffer)
    return paragraphs



class MyVectorDBConnector:
    def __init__(self, collection_name, embedding_fn):
        """内存模式"""
        # chroma_client = chromadb.Client(Settings(allow_reset=True))
        """本地模式"""
        chroma_client = chromadb.PersistentClient(path=f'./chroma/{collection_name}')

        # chroma_client.reset()

        # 创建一个 collection
        self.collection = chroma_client.get_or_create_collection(name=collection_name)
        self.embedding_fn = embedding_fn

    def add_documents(self, documents):
        '''向collection中添加文档与向量'''
        self.collection.add(
            embeddings=self.embedding_fn(documents),  # 每个文档的向量
            documents=documents,  # 文档的原文
            ids=[f"id{i}" for i in range(len(documents))]  # 每个文档的 id
        )


    def search(self, query, top_n):
        '''检索向量数据库'''
        results = self.collection.query(
            query_embeddings=self.embedding_fn([query]),
            n_results=top_n
        )
        return results


if __name__ == '__main__':
    paragraphs = extract_text_from_pdf("/Users/yuejunzhang/Desktop/AI_LLM06/day07Naive RAG Base/llama2.pdf", min_line_length=10)
    print('paragraphs:', paragraphs)

    # 创建一个向量数据库对象
    vector_db = MyVectorDBConnector("demoaibook", get_embeddings)

    # 向向量数据库中添加文档
    vector_db.add_documents(paragraphs)

    user_query = "llama 2有多少参数?"
    results = vector_db.search(user_query, 3)
    print('results: ',results)


    for para in results['documents'][0]:
        print(para+"\n")


#### 3.4.1 主流向量数据库

<img src="./img/vectordb.png" width="80%">

- FAISS: Meta 开源的向量检索引擎 https://github.com/facebookresearch/faiss
- Pinecone: 商用向量数据库，只有云服务 https://www.pinecone.io/
- Milvus: 开源向量数据库，同时有云服务 https://milvus.io/
- Weaviate: 开源向量数据库，同时有云服务 https://weaviate.io/
- Qdrant: 开源向量数据库，同时有云服务 https://qdrant.tech/
- PGVector: Postgres 的开源向量检索引擎 https://github.com/pgvector/pgvector
- RediSearch: Redis 的开源向量检索引擎 https://github.com/RediSearch/RediSearch
- ElasticSearch 也支持向量检索 https://www.elastic.co/enterprise-search/vector-search


### 4、 HuggingFace向量模型本地部署

huggingface:https://huggingface.co/

魔搭社区:https://www.modelscope.cn/my/overview

In [ ]:
# 模型下载
from modelscope import snapshot_download
model_dir = snapshot_download('AI-ModelScope/bge-large-zh-v1.5')

## 四、基于向量检索的RAG实现公司HR制度智能问答系统

In [ ]:
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI
from pdfminer.high_level import extract_pages
from pdfminer.layout import LTTextContainer
import chromadb
from chromadb.config import Settings
from docx import Document
client = OpenAI()
prompt_template = """
你是一个问答机器人。
你的任务是根据下述给定的已知信息回答用户问题。
确保你的回复完全依据下述已知信息。不要编造答案。
如果下述已知信息不足以回答用户的问题，请直接回复"我无法回答您的问题"。

已知信息:
__INFO__

用户问：
__QUERY__

请用中文回答用户问题。
"""

def extract_text_from_pdf(filename, page_numbers=None, min_line_length=1):
    '''从 PDF 文件中（按指定页码）提取文字'''
    paragraphs = []
    buffer = ''
    full_text = ''
    # 提取全部文本
    for i, page_layout in enumerate(extract_pages(filename)):
        # 如果指定了页码范围，跳过范围外的页
        if page_numbers is not None and i not in page_numbers:
            continue
        for element in page_layout:
            if isinstance(element, LTTextContainer):
                full_text += element.get_text() + '\n'

    # 按空行分隔，将文本重新组织成段落
    lines = full_text.split('\n')
    for text in lines:
        if len(text) >= min_line_length:
            buffer += (' '+text) if not text.endswith('-') else text.strip('-')
        elif buffer:
            paragraphs.append(buffer)
            buffer = ''
    if buffer:
        paragraphs.append(buffer)
    return paragraphs

def extract_text_from_docx(filename, min_line_length=1):
    '''从 DOCX 文件中提取文字'''
    paragraphs = []
    buffer = ''
    full_text = ''
    # 打开并读取文档
    doc = Document(filename)
    # 提取全部文本
    for para in doc.paragraphs:
        full_text += para.text + '\n'

    # 按空行分隔，将文本重新组织成段落
    lines = full_text.split('\n')
    for line in lines:
        if len(line) >= min_line_length:
            buffer += (' ' + line) if not line.endswith('-') else line.strip('-')
        elif buffer:
            paragraphs.append(buffer)
            buffer = ''
    if buffer:
        paragraphs.append(buffer)
    return paragraphs

# 使用示例
docx_filename = "人事管理流程.docx"
paragraphs = extract_text_from_docx(docx_filename, min_line_length=10)


# paragraphs = extract_text_from_pdf("人事管理流程.pdf", page_numbers=[
#                                    2, 3], min_line_length=10)


def get_completion(prompt, model="gpt-3.5-turbo"):
    '''封装 openai 接口'''
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0,  # 模型输出的随机性，0 表示随机性最小
    )
    return response.choices[0].message.content
def build_prompt(prompt_template, **kwargs):
    '''将 Prompt 模板赋值'''
    prompt = prompt_template
    for k, v in kwargs.items():
        if isinstance(v, str):
            val = v
        elif isinstance(v, list) and all(isinstance(elem, str) for elem in v):
            val = '\n'.join(v)
        else:
            val = str(v)
        prompt = prompt.replace(f"__{k.upper()}__", val)
    return prompt
class MyVectorDBConnector:
    def __init__(self, collection_name, embedding_fn):
        chroma_client = chromadb.Client(Settings(allow_reset=True))
        # 为了演示，实际不需要每次 reset()
        # chroma_client.reset()

        # 创建一个 collection
        self.collection = chroma_client.get_or_create_collection(name=collection_name)
        self.embedding_fn = embedding_fn

    def add_documents(self, documents):
        '''向 collection 中添加文档与向量'''
        self.collection.add(
            embeddings=self.embedding_fn(documents),  # 每个文档的向量
            documents=documents,  # 文档的原文
            ids=[f"id{i}" for i in range(len(documents))]  # 每个文档的 id
        )

    def search(self, query, top_n):
        '''检索向量数据库'''
        results = self.collection.query(
            query_embeddings=self.embedding_fn([query]),
            n_results=top_n
        )
        return results
def get_embeddings(texts, model="text-embedding-3-large"):
    '''封装 OpenAI 的 Embedding 模型接口'''
    data = client.embeddings.create(input=texts, model=model).data
    return [x.embedding for x in data]
# 创建一个向量数据库对象
vector_db = MyVectorDBConnector("demo", get_embeddings)

# 向向量数据库中添加文档
vector_db.add_documents(paragraphs)
class RAG_Bot:
    def __init__(self, vector_db, llm_api, n_results=2):
        self.vector_db = vector_db
        self.llm_api = llm_api
        self.n_results = n_results

    def chat(self, user_query):
        # 1. 检索
        search_results = self.vector_db.search(user_query, self.n_results)
        print('search_results:',search_results)
        # 2. 构建 Prompt
        prompt = build_prompt(
            prompt_template, info=search_results['documents'][0], query=user_query)

        print('prompt:',prompt)
        # 3. 调用 LLM
        response = self.llm_api(prompt)
        return response

# 创建一个RAG机器人
bot = RAG_Bot(
    vector_db,
    llm_api=get_completion
)
user_query = "视为不符合录用条件的情形有哪些?"
response = bot.chat(user_query)
print(response)